### What is Query Decomposition?
Query decomposition is the process of taking a complex, multi-part question and breaking it into simpler, atomic sub-questions that can each be retrieved and answered individually.

#### Why Use Query Decomposition?

- Complex queries often involve multiple concepts

- LLMs or retrievers may miss parts of the original question

- It enables multi-hop reasoning (answering in steps)

- Allows parallelism (especially in multi-agent frameworks)

In [1]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.runnables import RunnableSequence

In [2]:
loader = TextLoader("langchain_crewai_dataset.txt")

raw_doc = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)

chunks = text_splitter.split_documents(raw_doc)

In [3]:
chunks[0:5]

[Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain is an open-source framework designed for developing applications powered by large language models (LLMs). It simplifies the process of building, managing, and scaling complex chains of thought by abstracting prompt management, retrieval, memory, and agent orchestration. Developers can use'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='and agent orchestration. Developers can use LangChain to create end-to-end pipelines that connect LLMs with tools, APIs, vector databases, and other knowledge sources. (v1)'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='At the heart of LangChain lies the concept of chains, which are sequences of calls to LLMs and other tools. Chains can be simple, such as a single prompt fed to an LLM, or complex, involving multiple conditionally executed steps. LangChain makes it easy to compose and reuse chains using st

In [4]:
embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embedding)
retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 4, "lambda_mult": 0.7})

In [5]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

llm = init_chat_model(
    model="groq/compound-mini",
    model_provider="groq",
    api_key=os.getenv("GROQ_API_KEY"),
)
llm

ChatGroq(profile={}, client=<groq.resources.chat.completions.Completions object at 0x000001B27B79A810>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001B2002F3BC0>, model_name='groq/compound-mini', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [6]:
# Query decomposition prompt template
decomposition_prompt = PromptTemplate.from_template("""
You are an AI assistant. Decompose the following complex question into 2 to 4 smaller sub-questions for better document retrieval.

Question: "{question}"

Sub-questions:
""")

decomposition_chain = decomposition_prompt | llm | StrOutputParser()

In [7]:
query = "How does LangChain use memory and agents compared to CrewAI?"

decomposition_question = decomposition_chain.invoke({"question": query})

In [8]:
decomposition_question

'**Sub‑questions**\n\n1. What memory mechanisms does LangChain offer, and how are they typically used in LangChain applications?  \n2. What types of agents does LangChain provide, and how do they operate (e.g., tool‑calling, planning, execution)?  \n3. What memory mechanisms does CrewAI provide, and how are they employed within CrewAI workflows?  \n4. In what ways do LangChain’s memory and agent implementations differ from CrewAI’s (e.g., architecture, capabilities, extensibility, typical use‑cases)?'

In [9]:
sample = decomposition_question.strip().split("\n")
sample

['**Sub‑questions**',
 '',
 '1. What memory mechanisms does LangChain offer, and how are they typically used in LangChain applications?  ',
 '2. What types of agents does LangChain provide, and how do they operate (e.g., tool‑calling, planning, execution)?  ',
 '3. What memory mechanisms does CrewAI provide, and how are they employed within CrewAI workflows?  ',
 '4. In what ways do LangChain’s memory and agent implementations differ from CrewAI’s (e.g., architecture, capabilities, extensibility, typical use‑cases)?']

In [10]:
sample_sub_questions = [q.strip("-1234567890. ").strip() for q in sample if q.strip()]
sample_sub_questions

['**Sub‑questions**',
 'What memory mechanisms does LangChain offer, and how are they typically used in LangChain applications?',
 'What types of agents does LangChain provide, and how do they operate (e.g., tool‑calling, planning, execution)?',
 'What memory mechanisms does CrewAI provide, and how are they employed within CrewAI workflows?',
 'In what ways do LangChain’s memory and agent implementations differ from CrewAI’s (e.g., architecture, capabilities, extensibility, typical use‑cases)?']

In [18]:
qa_prompt = PromptTemplate.from_template("""
Use the context below to answer the question.
                                         
Context:
{context}
                                         
Question: {input}
""")

qa_chain = create_stuff_documents_chain(llm=llm, prompt=qa_prompt)

In [19]:
def full_query_decomposition_rag_pipeline(user_query:str):
    # Decompose the query
    sub_questions = decomposition_chain.invoke({"question": user_query})
    sub_questions_list = [q.strip("-1234567890. ").strip() for q in sub_questions.split("\n") if q.strip()]

    results = []
    for subq in sub_questions_list:
        docs = retriever.invoke(subq)
        result = qa_chain.invoke({"input": subq, "context": docs})
        results.append(f"Q: {subq}\nA: {result}")

    return "\n\n".join(results)

In [20]:
query = "How does LangChain use memory and agents compared to CrewAI?"
final_answer = full_query_decomposition_rag_pipeline(query)
print("Final Answer:\n")
print(final_answer)

Final Answer:

Q: **Sub‑questions**
A: Here are some useful sub‑questions you could ask about the material in the context:

1. **Knowledge injection**
   - How does injecting fetched knowledge into the LLM prompt improve accuracy?
   - In what ways does knowledge injection help reduce hallucinations in language models?
   - What are the typical formats or structures used to embed external knowledge into prompts?

2. **CrewAI’s agent context‑sharing**
   - What is “agent context‑sharing” in CrewAI and how is it implemented?
   - How do agents pass intermediate data to one another in a structured manner?
   - What emergent behaviors (e.g., delegation, consultation, review) arise from this context‑sharing?
   - How does context‑sharing enable both horizontal scaling (more agents) and vertical scaling (deeper reasoning)?

3. **LangChain prompt engineering**
   - What templating capabilities does LangChain provide for prompt creation?
   - How can input variables and formatting options be u